In [9]:
# импорт зависимосте

import sys
from pathlib import Path

import pandas as pd
import numpy as np
import xarray as xr

project_root = Path.cwd().resolve()
while not (project_root / "userpwd.py").exists():
    project_root = project_root.parent
    assert project_root != project_root.parent, "Could not locate repo root with userpwd.py"

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import psycopg
import userpwd

In [10]:
def req(y0, y1):
    return """
                       select dt, speed, temperature, density
                       from rss.ace_at_earth_1h
                       where (
                       ace_at_earth_1h.dt >= DATE '{0}-01-01'
                       and
                       ace_at_earth_1h.dt <= DATE '{1}-01-01'
                       )
""".format(y0, y1)

In [11]:
years = list(range(2018, 2021))
years

[2018, 2019, 2020]

In [12]:
reqs = []

for i in range(len(years) - 1):
    reqs.append(req(years[i], years[i + 1]))
reqs

["\n                       select dt, speed, temperature, density\n                       from rss.ace_at_earth_1h\n                       where (\n                       ace_at_earth_1h.dt >= DATE '2018-01-01'\n                       and\n                       ace_at_earth_1h.dt <= DATE '2019-01-01'\n                       )\n",
 "\n                       select dt, speed, temperature, density\n                       from rss.ace_at_earth_1h\n                       where (\n                       ace_at_earth_1h.dt >= DATE '2019-01-01'\n                       and\n                       ace_at_earth_1h.dt <= DATE '2020-01-01'\n                       )\n"]

In [13]:
res = []

conn = psycopg.connect(
    host="213.131.1.41", user="selector", dbname="smdc", password=userpwd.userpwd_postgre
)

In [14]:
with conn.cursor() as cur:
    for req in reqs:
        print(req)
        for row in cur.execute(req):
            res.append(row)


                       select dt, speed, temperature, density
                       from rss.ace_at_earth_1h
                       where (
                       ace_at_earth_1h.dt >= DATE '2018-01-01'
                       and
                       ace_at_earth_1h.dt <= DATE '2019-01-01'
                       )


                       select dt, speed, temperature, density
                       from rss.ace_at_earth_1h
                       where (
                       ace_at_earth_1h.dt >= DATE '2019-01-01'
                       and
                       ace_at_earth_1h.dt <= DATE '2020-01-01'
                       )



In [15]:
df = pd.DataFrame(np.array(res), columns=["date", "speed", "temperature", "density"])

df["date"] = pd.to_datetime(df["date"])

df.set_index("date", inplace=True)

df

,speed,temperature,density
date,,,
2018-04-05 12:00:00,368.016667,56337.5,6.435833
2018-04-05 09:00:00,361.151667,47924.166667,6.616667
2018-04-05 11:00:00,369.775,67548.333333,5.935
2018-04-05 10:00:00,361.675,51895.833333,6.3775
2018-04-06 01:00:00,428.573333,109046.666667,4.064167
...,...,...,...
2019-12-31 21:00:00,309.123214,33016.964286,3.03125
2019-12-31 15:00:00,294.683898,28016.101695,3.387288
2019-12-31 10:00:00,303.099167,23614.166667,3.18


In [16]:
df.to_parquet("../Data/ACE At Earth 1h.parquet")